<a href="https://colab.research.google.com/github/kishorenk137/daa-1/blob/main/extra_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def rabin_karp_2d(text, pattern):
    text_rows = len(text)
    text_cols = len(text[0])
    pat_rows = len(pattern)
    pat_cols = len(pattern[0])

    # Base and Prime for the rolling hash
    BASE = 101
    PRIME = 1000005001

    # Precompute (BASE ** (pat_cols - 1)) % PRIME for horizontal sliding
    base_col_power = 1
    for _ in range(pat_cols - 1):
        base_col_power = (base_col_power * BASE) % PRIME

    comparison_count = 0
    match_positions = []

    # 1. Compute the hash value of the pattern
    pattern_hash = 0
    for r in range(pat_rows):
        row_hash = 0
        for c in range(pat_cols):
            row_hash = (row_hash * BASE + ord(pattern[r][c])) % PRIME
        pattern_hash = (pattern_hash * BASE + row_hash) % PRIME

    # 2. Precompute horizontal row hashes for every window of width `pat_cols`
    # row_hashes[r][c] will store the hash of text[r][c : c + pat_cols]
    row_hashes = [[0] * (text_cols - pat_cols + 1) for _ in range(text_rows)]

    for r in range(text_rows):
        # Initial hash for the first window in row `r`
        current_hash = 0
        for c in range(pat_cols):
            current_hash = (current_hash * BASE + ord(text[r][c])) % PRIME
        row_hashes[r][0] = current_hash

        # Roll horizontally across the rest of the row
        for c in range(1, text_cols - pat_cols + 1):
            # Remove high-order column, shift, and add new low-order column
            current_hash = (current_hash - ord(text[r][c - 1]) * base_col_power) % PRIME
            current_hash = (current_hash * BASE + ord(text[r][c + pat_cols - 1])) % PRIME
            row_hashes[r][c] = (current_hash + PRIME) % PRIME # Ensure positive

    # Precompute (BASE ** (pat_rows - 1)) % PRIME for vertical sliding
    base_row_power = 1
    for _ in range(pat_rows - 1):
        base_row_power = (base_row_power * BASE) % PRIME

    # 3. Slide vertically over the row hashes to find the 2D window hashes
    for c in range(text_cols - pat_cols + 1):
        # Calculate initial 2D hash for the top window at column `c`
        current_2d_hash = 0
        for r in range(pat_rows):
            current_2d_hash = (current_2d_hash * BASE + row_hashes[r][c]) % PRIME

        # Check first window at (0, c)
        if current_2d_hash == pattern_hash:
            # Hash match found! Verify characters to confirm (spares false positives)
            if verify_match(text, pattern, 0, c):
                match_positions.append((0, c))
            comparison_count += (pat_rows * pat_cols)

        # Roll vertically down through the remaining rows
        for r in range(1, text_rows - pat_rows + 1):
            current_2d_hash = (current_2d_hash - row_hashes[r - 1][c] * base_row_power) % PRIME
            current_2d_hash = (current_2d_hash * BASE + row_hashes[r + pat_rows - 1][c]) % PRIME
            current_2d_hash = (current_2d_hash + PRIME) % PRIME

            if current_2d_hash == pattern_hash:
                comparison_count += (pat_rows * pat_cols)
                if verify_match(text, pattern, r, c):
                    match_positions.append((r, c))

    return match_positions, comparison_count


def verify_match(text, pattern, start_r, start_c):
    """Helper function to perform exact element comparisons on a hash match."""
    for r in range(len(pattern)):
        for c in range(len(pattern[0])):
            if text[start_r + r][start_c + c] != pattern[r][c]:
                return False
    return True

# --- Test Data ---
text_matrix = [
    ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'],
    ['B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A'],
    ['A', 'B', 'X', 'Y', 'Z', 'B', 'A', 'B', 'A', 'B'],
    ['B', 'A', 'W', 'P', 'Q', 'A', 'B', 'A', 'B', 'A'],
    ['A', 'B', 'M', 'N', 'O', 'B', 'A', 'B', 'A', 'B'],
    ['B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A'],
    ['A', 'B', 'A', 'B', 'A', 'B', 'X', 'Y', 'Z', 'B'],
    ['B', 'A', 'B', 'A', 'B', 'A', 'W', 'P', 'Q', 'A'],
    ['A', 'B', 'A', 'B', 'A', 'B', 'M', 'N', 'O', 'B'],
    ['B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A']
]

pattern_matrix = [
    ['X', 'Y', 'Z'],
    ['W', 'P', 'Q'],
    ['M', 'N', 'O']
]

matches, comparisons = rabin_karp_2d(text_matrix, pattern_matrix)
print(f"Pattern found at top-left coordinates: {matches}")
print(f"Total element-level comparisons made: {comparisons}")

Pattern found at top-left coordinates: [(2, 2), (6, 6)]
Total element-level comparisons made: 18
